
Goal: load a CSV into SQLite, pull its schema, and get one working function
that turns a question into SQL and runs it.

In [ ]:
import pandas as pd
import sqlite3
from groq import Groq

client  = Groq()   # reads GROQ_API_KEY from env automatically
MODEL   = 'llama-3.3-70b-versatile'
DB_PATH = 'data.db'

In [ ]:
def load_csv_to_sqlite(csv_path, db_path=DB_PATH, table_name='data'):
    df = pd.read_csv(csv_path)
    # clean column names: lowercase + underscores
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    conn = sqlite3.connect(db_path)
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    conn.close()
    print(f'Loaded {len(df)} rows into table: {table_name}')
    return table_name

# swap path below to your own CSV
table_name = load_csv_to_sqlite('../sample_data/sales_sample.csv')

In [ ]:
def get_schema(table_name, db_path=DB_PATH):
    conn   = sqlite3.connect(db_path)
    cols   = conn.execute(f'PRAGMA table_info({table_name})').fetchall()
    sample = conn.execute(f'SELECT * FROM {table_name} LIMIT 3').fetchall()
    conn.close()
    col_lines    = [f'  - {c[1]} ({c[2]})' for c in cols]
    sample_lines = [str(row) for row in sample]
    return (
        f'Table: {table_name}\nColumns:\n' + '\n'.join(col_lines)
        + '\nSample rows:\n' + '\n'.join(sample_lines)
    )

schema_text = get_schema(table_name)
print(schema_text)

Nl to sql via groq 


In [ ]:
def ask_question(question, schema, model=MODEL):
    prompt = (
        'You are a SQLite expert. Write ONE SELECT query that answers the question.\n'
        'Return ONLY raw SQL — no markdown fences, no explanation.\n\n'
        f'{schema}\n\nQuestion: {question}\nSQL:'
    )
    resp = client.chat.completions.create(
        model=model,
        max_tokens=300,
        messages=[{'role': 'user', 'content': prompt}],
    )
    sql = resp.choices[0].message.content.strip()
    # strip markdown fences if the model adds them
    return sql.strip('`').removeprefix('sql').strip()

In [ ]:
def run_sql(sql, db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    df   = pd.read_sql_query(sql, conn)
    conn.close()
    return df

In [ ]:
questions = [
    'What is the total revenue by region?',
    'Which product had the highest quantity sold overall?',
    'List the top 3 customers by total spend',
]

for q in questions:
    print(f'Q: {q}')
    sql = ask_question(q, schema_text)
    print(f'SQL: {sql}')
    try:
        print(run_sql(sql))
    except Exception as e:
        print(f'ERROR: {e}')
    print('-' * 50)